# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from anthropic import Anthropic

In [17]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

if api_key and api_key.startswith('sk-ant-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'claude-haiku-4-5-20251001'
claude = Anthropic()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [18]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [19]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [24]:
def select_relevant_links(url):
    response = claude.messages.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        max_tokens=1000,
        system=link_system_prompt,
        # response_format={"type": "json_object"}
    )
    result = response.content[0].text.strip()
    start = result.find('{')
    links, _ = json.JSONDecoder().raw_decode(result[start:])
    return links
    

In [25]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog/posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum/courses',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'products', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'products', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'products', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'products', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'}]}

In [28]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = claude.messages.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        max_tokens=1000,
        system=link_system_prompt
    )
    result = response.content[0].text.strip()
    start = result.find('{')
    links, _ = json.JSONDecoder().raw_decode(result[start:])
    print(f"Found {len(links['links'])} relevant links")
    return links

In [29]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling claude-haiku-4-5-20251001
Found 10 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'curriculum/courses',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog/posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'product', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'product', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'}]}

In [30]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 17 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'social media', 'url': 'https://github.com/huggingface'},
  {'type': 'social media', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'm

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [31]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [32]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 17 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Edge0/Audio8-ASR-Infinite
Updated
7 days ago
•
26.7k
•
1.98k
convaiinnovations/laya
Updated
7 days ago
•
4.72k
XingChen-AGI/TeleOCR
Updated
2 days ago
•
30.4k
•
1.12k
abenzerps/Qwen-Image-2.1-Uncensored-GGUF
Updated
3 days ago
•
1.23M
•
2.61k
Contrastive-LM/CLM-v0.1-8B


In [42]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [34]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [35]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 18 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nEdge0/Audio8-ASR-Infinite\nUpdated\n7 days ago\n•\n26.7k\n•\n1.98k\nconvaiinnovations/laya\nUpdated\n7 days ago\n•\n

In [36]:
def create_brochure(company_name, url):
    response = claude.messages.create(
        model="claude-haiku-4-5",
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        max_tokens=1000,
        system=brochure_system_prompt
    )
    result = response.content[0].text.strip()
    display(Markdown(result))

In [37]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 17 relevant links


# Hugging Face: The AI Community Building the Future

## About Hugging Face

Hugging Face is the central platform where the machine learning community collaborates on artificial intelligence models, datasets, and applications. It serves as the home of machine learning, enabling creators, researchers, and developers to discover, build, and deploy AI solutions together.

## Core Offerings

### Models
Browse and contribute to a library of over 2 million pre-trained machine learning models. Models cover diverse tasks including:
- Text generation and processing
- Image generation and analysis
- Audio processing and speech recognition
- Video generation
- And 44+ additional specialized tasks

Models are available across multiple frameworks including PyTorch, TensorFlow, JAX, and more, with varying parameter sizes from under 1 billion to over 500 billion parameters.

### Datasets
Access a comprehensive collection of over 500,000 datasets supporting machine learning projects. Datasets range from specialized medical conversation data to comprehensive arxiv archives, enabling researchers and developers to find exactly what they need for training and fine-tuning.

### Spaces & Applications
Deploy and showcase AI applications through Spaces, with over 1 million applications hosted on the platform. Users can run interactive demos, benchmark different models, and explore real-world AI applications from video generation to image editing.

### Enterprise Solutions
- **Hugging Face PRO** - Enhanced access for individual users
- **Enterprise Support** - Dedicated support for organizations
- **Inference Endpoints** - Deploy models with managed infrastructure
- **Storage Buckets** - Scalable storage for ML assets

## Community & Collaboration

Hugging Face fosters an active global community through:
- Discord and Forum for peer support
- GitHub integration for open-source development
- Daily Papers and Blog for staying current with AI research
- Collections and Organizations for collaborative work
- HuggingChat for interactive AI experiences

## Why Hugging Face

- **Massive Scale**: 2M+ models, 500K+ datasets, 1M+ applications
- **Open Collaboration**: Built on community contributions and shared knowledge
- **Accessibility**: Tools and resources for beginners to advanced researchers
- **Flexibility**: Support for multiple frameworks and use cases
- **Innovation**: Constantly trending with the latest AI breakthroughs

Whether you're a researcher pushing the boundaries of AI, a developer building the next generation of applications, or an organization seeking enterprise ML solutions, Hugging Face provides the infrastructure and community to make it happen.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [40]:
def stream_brochure(company_name, url):
    stream = claude.messages.create(
        model="claude-haiku-4-5",
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True,
        max_tokens=1000,
        system=brochure_system_prompt
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.type == 'content_block_delta':
            response += chunk.delta.text or ''
            display_handle.update(Markdown(response))

In [41]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 17 relevant links


# Hugging Face Brochure

## About Hugging Face

Hugging Face is the collaboration platform for the machine learning community, serving as the home of machine learning where developers, researchers, and organizations create, discover, and collaborate on AI better together.

## What We Do

Hugging Face operates a comprehensive platform that enables the AI community to build the future through collaboration. The platform hosts and provides access to:

- **2M+ Machine Learning Models** – A massive repository of pre-trained models across various AI tasks
- **500k+ Datasets** – High-quality datasets for training and evaluation
- **1M+ AI Applications** – Ready-to-use spaces and applications built by the community
- **Storage and Infrastructure** – Tools including Inference Endpoints, Storage Buckets, and inference providers for deployment

## Core Features

The Hugging Face Hub is a centralized collaboration space where the machine learning community can:

- Host and collaborate on models, datasets, and applications
- Discover trending AI applications and cutting-edge models
- Deploy AI applications through Spaces
- Access comprehensive documentation and learning resources
- Participate in the active community through Discord, forums, and GitHub

## Solutions & Services

- **Hugging Face PRO** – Premium membership for individual users
- **Team & Enterprise** – Custom solutions for organizations
- **Enterprise Support** – Dedicated assistance for business deployments
- **HuggingChat** – An AI chat interface powered by the platform
- **Hardware Resources** – Computing infrastructure for model training and deployment

## Community & Culture

Hugging Face is built on collaboration and democratization of AI. The platform hosts a vibrant global community of developers, researchers, and organizations who contribute models, share datasets, and build applications together. With active communities on Discord and GitHub, Hugging Face fosters knowledge sharing, peer support, and collective innovation in machine learning.

In [44]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 17 relevant links


# 🤗 HUGGING FACE: WHERE AI GOES TO COLLABORATE (AND OCCASIONALLY ARGUE ABOUT HYPERPARAMETERS)

## Welcome to the World's Friendliest AI Community

Think of Hugging Face as the GitHub for machine learning, if GitHub hugged you back. We're the platform where 2M+ AI models, 500K+ datasets, and 1M+ applications live in harmonious coexistence—a digital utopia where researchers, engineers, and aspiring AI enthusiasts gather to build the future without requiring a PhD in quantum physics (though it doesn't hurt).

## What We Do

**We host stuff. Big stuff. Smart stuff.**

- **2 Million+ Models** – From audio ASR wizards to image generation maestros, we've got more pretrained models than you have unfinished projects on your laptop
- **500K+ Datasets** – Everything from doctor-patient conversations to complete arXiv papers. Yes, we catalogued the entire internet's nerdiness
- **1 Million+ Applications** – Real, working AI apps built by real humans in "Spaces," because sometimes you just want to *use* the AI instead of talking about it at parties

Our playground includes video generation with synchronized soundtracks, image editing from text descriptions, and benchmarks tracking the latest AI breakthroughs. It's like an arcade, but instead of winning stuffed animals, you win peer recognition and the ability to brag about your model on Discord.

## The Platform That Gets It

Unlike other ML platforms that feel like they were designed by people who've never spoken to humans, Hugging Face actually *understands* the community:

- **Free tier that doesn't suck** – Host your models, datasets, and spaces without selling your kidney
- **Enterprise solutions** – For companies with more ambition (and budget) than a startup's first iteration
- **HuggingChat** – Because sometimes you want to chat with AI without setting up your own 70B parameter monster
- **Learning resources** – Docs, Discord, Forum, and Daily Papers to keep you from feeling like a complete newbie

## Who Uses Hugging Face?

Everyone. Literally.

- **Researchers** pushing the boundaries of what's possible
- **Students** learning ML without breaking the bank or their laptops
- **Companies** from startups to enterprises building production AI systems
- **Random Enthusiasts** training their first models at 2 AM with optimistic vibes and realistic caffeine consumption

## Why We're Different

While other platforms whisper about "enterprise alignment" and "synergistic paradigm shifts," we just… build stuff that works and make it easy for everyone else to use it. Our community doesn't gatekeep—we celebrate when someone's GGUF model gets 1.3M downloads. We host trending models like audio recognition systems, image generation tools, and specialized OCR models updated mere hours ago.

We're basically the "everyone gets a trophy" of AI platforms, except your trophy actually works and runs on actual hardware.

## Join Us

Whether you're here to:
- Download a model that took someone 6 months to train
- Publish your own creation to the 500K+ dataset collection
- Deploy an AI app without managing servers
- Lurk and learn from the smartest people in ML

...you're in the right place.

**The future of AI isn't being built in some secret lab. It's being built here, in public, with you.**

Now stop reading this brochure and go build something cool. 🚀

---

*Hugging Face: Where Models Thrive, Communities Collaborate, and Your GPU Finally Has a Purpose*

In [53]:
insights_system_prompt = """
You are an assistant that extracts key business insights from a company brochure and turns it to german language.
Return a JSON object with these fields:
- company_strengths: list of 3-5 key strengths
- ideal_customers: who they serve best
- hiring_focus: what types of roles they're hiring for
"""

In [54]:
def get_insights_user_prompt(brochure_content):
    return f"""
            Based on this brochure about a company, extract key insights in JSON format: {brochure_content}
            """

In [55]:
def extract_insights(brochure_content):
    response = claude.messages.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": get_insights_user_prompt(brochure_content)}
        ],
        max_tokens=1000,
        system=insights_system_prompt,
    )
    result = response.content[0].text.strip()
    start = result.find('{')
    insights, _ = json.JSONDecoder().raw_decode(result[start:])
    return insights


In [56]:
def full_company_analysis(company_name, url):
    brochure_response = claude.messages.create(
        model="claude-haiku-4-5",
        messages=[
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        max_tokens=1000,
        system=brochure_system_prompt)
    brochure_content = brochure_response.content[0].text.strip()
    insights = extract_insights(brochure_content)
    return brochure_content, insights

full_company_analysis("HuggingFace", "https://huggingface.co")


Selecting relevant links for https://huggingface.co by calling claude-haiku-4-5-20251001
Found 15 relevant links


("# 🤗 HUGGING FACE: A BROCHURE\n\n## *Where AI Goes to Make Friends*\n\n---\n\n## WHAT IS HUGGING FACE, ANYWAY?\n\nThink of it as GitHub, but for AI models. Except instead of developers arguing about tabs vs. spaces, thousands of machine learning enthusiasts collaborate on building the future while actually getting along. Revolutionary? Probably. Confusing at first? Absolutely.\n\nWe're the home of:\n- **2M+ AI models** (some good, some experimental, all fascinating)\n- **500k+ datasets** (the fuel that makes AI go vroom)\n- **1M+ AI applications** (Spaces where people do genuinely wild things with machine learning)\n\nPlus HuggingChat, Discord, forums, and enough documentation to keep your eyes glazed over for weeks.\n\n---\n\n## THE VIBE CHECK\n\n**Open Source Energy, Professional Execution.** We're genuinely about democratizing AI—making powerful models accessible whether you're a solo researcher in your garage or a Fortune 500 company. The community actually uses our stuff. Wild co

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>